# GP03 — CSV to GIS Dataset

CSV → DataFrame → GeoDataFrame → CRS → QA → Export

Το CSV είναι tabular source. Το `points_from_xy()` προσθέτει geometry, το CRS δηλώνει την ερμηνεία των coordinates και το re-read μετά το export επιβεβαιώνει ότι παραδόθηκε GIS dataset.

## 1. CSV schema

Ελέγχουμε την πηγή πριν δημιουργήσουμε geometry.

In [ ]:
import pandas as pd
import geopandas as gpd
from pathlib import Path
df = pd.read_csv("data/gp03/city_assets.csv", sep=",", encoding="utf-8")
print("Rows:", len(df))
print("Columns:", list(df.columns))
print(df.dtypes)
assert df.isna().sum().sum() == 0
assert df["asset_id"].is_unique

## 2. Geometry από longitude/latitude

In [ ]:
gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df["longitude"], df["latitude"]), crs="EPSG:4326")
print(gdf.geometry.geom_type.value_counts())
print("Bounds:", gdf.total_bounds)
gdf.plot(figsize=(7, 6), color="navy")

## 3. CRS: set_crs και to_crs

Το set_crs δηλώνει το γνωστό CRS. Το to_crs δημιουργεί μετασχηματισμένο αντίγραφο.

In [ ]:
assert gdf.crs.to_epsg() == 4326
projected = gdf.to_crs("EPSG:2100")
assert projected.crs.to_epsg() == 2100
assert gdf.crs.to_epsg() == 4326

## 4. GeoPython QA

In [ ]:
assert len(gdf) == len(df)
assert len(gdf) > 0
assert gdf.geometry.geom_type.eq("Point").all()
assert gdf.geometry.notna().all()
assert gdf.drop(columns="geometry").isna().sum().sum() == 0
assert gdf.geometry.is_valid.all()
assert gdf["longitude"].between(-180, 180).all()
assert gdf["latitude"].between(-90, 90).all()
assert gdf.total_bounds.shape == (4,)
print("Rows:", len(gdf), "CRS:", gdf.crs)

## 5. Export σε δύο formats

In [ ]:
output_dir = Path("outputs/gp03")
output_dir.mkdir(parents=True, exist_ok=True)
geojson_path = output_dir / "city_assets.geojson"
gpkg_path = output_dir / "city_assets.gpkg"
gdf.to_file(geojson_path, driver="GeoJSON")
gdf.to_file(gpkg_path, layer="assets", driver="GPKG")
check = gpd.read_file(gpkg_path, layer="assets")
assert len(check) == len(gdf)
assert check.crs is not None
print("GeoJSON και GeoPackage export OK")

## Mini Challenge — Ενδεικτική λύση

In [ ]:
good = gdf[gdf["condition"] == "good"].copy()
assert len(good) > 0
assert (good["condition"] == "good").all()
good_path = output_dir / "good_assets.geojson"
good.to_file(good_path, driver="GeoJSON")
assert len(gpd.read_file(good_path)) == len(good)

## Τελικός έλεγχος

Το CSV έγινε GIS dataset με ελεγχόμενο schema, geometry, CRS και επαληθευμένο export.